# Занятие 2, демо 3. Затухание в половинной точности

Масштаб памяти - $1/(1-\gamma)$. Хотим 10 000 токенов - значит,
$\gamma=0.9999$. Смотрим, что остаётся от такой гаммы в разных форматах чисел.

## Гамма в fp16

In [ ]:
import torch

gamma = torch.tensor(1 - 1e-4, dtype=torch.float16)
print("гамма в fp16:", float(gamma))

In [ ]:
one = torch.tensor(1.0, dtype=torch.float16)
below = torch.nextafter(one, torch.tensor(0.0, dtype=torch.float16))
print("ближайшее к 1 снизу число в fp16:", float(below))

In [ ]:
print("вклад записи через 1500 шагов:", float(gamma) ** 1500)

## Та же гамма в fp32

In [ ]:
g32 = torch.tensor(1 - 1e-4, dtype=torch.float32)
print("гамма в fp32:", float(g32))
print("вклад записи через 1500 шагов:", float(g32) ** 1500)

In [ ]:
print("вклад записи через 10 000 шагов:", float(g32) ** 10000)

## Где начинается беда

In [ ]:
print(f"{'задумано':>10} {'в fp16':>14} {'масштаб':>14}")
for g in (0.9, 0.99, 0.999, 0.9999):
    h = float(torch.tensor(g, dtype=torch.float16))
    scale = f"{1 / (1 - h):.1f}" if h < 1 else "бесконечность"
    print(f"{g:>10} {h:>14.10f} {scale:>14}")

## bf16

In [ ]:
print("гамма в bf16:", float(torch.tensor(1 - 1e-4, dtype=torch.bfloat16)))

## Хранить логарифм

Логарифм гаммы считается до округления, в fp16 хранится уже он.

In [ ]:
import math

log_gamma = torch.tensor(math.log(1 - 1e-4), dtype=torch.float16)
print("логарифм гаммы в fp16:", float(log_gamma))
print("вклад записи через 1500 шагов:", float(torch.exp(1500 * log_gamma)))

## Что видно

В fp16 гамма $0.9999$ округляется в $1$: затухания нет, и ни одного сообщения
об ошибке. В bf16 - то же самое. Хранение логарифма маленькое затухание
сохраняет. Точность накопления сумм в настоящем коде проверяется отдельно.